In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv('GOOGLE_API_KEY')

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=1.0)

In [ ]:
pizzeria_menu = """
    --- Pizzeria Menu ---
    Regular Menu:
    - Margherita: $8.50
    - Pepperoni: $9.50
    - Vegetarian: $9.00
    - BBQ Chicken: $10.00
    
    Special Lunch Menu (Available everyday, 11:00 - 14:00):
    - Truffle Mushroom Pizza: $7.50
    - Spicy Salami Pizza: $7.00
    - Pesto Chicken Pizza: $7.50
    """

### Global state

In [ ]:
all_orders = {}# dict of key (order ID: str), values (dict: with keys: customer_name, distance, order_id, pizza_type, states (list), timestamps (list))
customer_dict = {} # dict of name, character, distance

### Customer Agents

In [ ]:
### create customer ordering agent prompt

# pydantic to force structured output (log event) produced by llm agent
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class OrderOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    pizza_type: str = Field(description="The kind of pizza the customer want to order, leave empty if customer does not order")
    timestamp: str = Field(description='The timestamp at which the activity is finished')

def create_order_agent(customer, start_time, end_time):
    name = customer['name']
    distance = customer['distance']
    character = customer['character']

    order_agent_prompt = """
    You are {name}, living {distance} km from Da Pipo Pizzeria. 
    {character}

    Do you want to order a pizza from Da Pipo Pizzeria?

    Here are the menu: 
    {pizzeria_menu}

    Your decision to order depends on multiple factors:  
    - **Weather** (e.g., rainy weather makes it a great time for pizza but may delay delivery)  
    - **Time** (e.g., you're lazier to cook on weekends)  
    - **Hunger level** (whether you're hungry right now)  
    - **Traffic** (heavy traffic may slow delivery)  
    - **Distance** (closer distance means faster delivery)  

    Considering these factors, decide whether to order pizza or not within the 10 minutes from {start_time} to {end_time}.
    Respond concisely with:  
    - **action**: "place order" if you would like to order pizza within the given time frame, or "next time" if otherwise. 
    - **pizza type**: The type of pizza you want (if ordering), otherwise leave it empty.
    - **timestamp**: The time at which you finalize your decision (The timestamp should be within the given time frame)  
    """
    prompt = PromptTemplate.from_template(order_agent_prompt)
    structured_llm = llm.with_structured_output(OrderOutput)
    customer_order_agent = prompt| structured_llm
    output = customer_order_agent.invoke({'name': name,
                                          'distance': distance,
                                          'character': character,
                                          'start_time': start_time,
                                          'end_time': end_time})
    return output

In [ ]:
# create customer cancelling agent

class CancelOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    timestamp: str = Field(description='The timestamp at which the activity is finished')

def create_cancel_agent(customer, order, start_time, end_time):
    name = customer['name']
    distance = customer['distance']
    character = customer['character']

    pizza_type = order['pizza_type']
    ordering_timestamp = order['timestamp'][0]

    customer_cancel_prompt = """
    You are {name}, living {distance} km from Da Pipo Pizzeria. 
    {character}

    You just placed an order for **{pizza_type}** at **{ordering_timestamp}**. The order is registered but not yet confirmed.  

    You should only **cancel the order** for special reasons, such as:  
    - **A sudden change in plans** (e.g., you need to leave home unexpectedly)    
    - **A mistake in your order** (e.g., you ordered the wrong pizza type)  

    Otherwise, you should wait for your order.  

    Now, decide your action in 10 minutes from {start_time} to {end_time}. and provide the following output:  

    - **action**: `"cancel order"` or `"wait for order"` 
    - **timestamp**: The time at which you finalize your decision (The timestamp should be within the given time frame).   

    Please choose your action.
    """
    prompt = PromptTemplate.from_template(customer_cancel_prompt)
    structured_llm = llm.with_structured_output(CancelOutput)
    customer_cancel_agent = prompt| structured_llm
    output = customer_cancel_agent.invoke({'name': name,
                                           'distance': distance,
                                           'character': character,
                                           'pizza_type': pizza_type,
                                           'ordering_timestamp': ordering_timestamp,
                                           'start_time': start_time,
                                           'end_time': end_time})

### Pizzeria Agent

In [ ]:
from typing import List
from pydantic import BaseModel, Field

# get order latest state from system state
from helpers import get_latest_states

class PizzeriaOutput(BaseModel):
    """Structured Output of System Agent"""
    order_id: str = Field(description="Order ID")
    action: str = Field(description="Action taken by the process participant")
    timestamp: str = Field(description="The timestamp at which the action is taken place")

class PizzeriaOutputMultiple(BaseModel):
    multiple_actions: List[PizzeriaOutput] = Field(description="Multiple actions taken by the process participant")

def pizzeria_agent(system_state, start_time, end_time):
    pizzeria_prompt = """
    You are the owner and sole worker of Da Pipo Pizzeria. Your pizzeria operates exclusively through online orders.  

    When a new order is placed, you must handle all necessary steps to fulfill it efficiently. These steps include: 

    - Confirming the order: Unconfirmed orders can be canceled. This usually takes 10-60 seconds.  
    - Preparing the order: Baking a pizza, which typically takes on average 20 minutes, but can takes longer if you make a mistake, e.g. bake wrong pizza and have to redo, but for sure it shouldn't happen so often.
    - Delivering the order: Delivery time depends on weather, traffic, and the distance to the customer, e.g. a distance of 10 km takes normally 15 mins but in busy traffic can takes up to nearly an hour.

    To optimize time, you can manage time in a efficient way and do multi-tasking, such as:  
    - Bake multiple pizzas simultaneously if you're notified of multiple orders at the same time.  
    - Confirm new orders or start baking another pizza while waiting for another one to be finished.  
    - Deliver multiple orders in one trip if they are nearby.  
    However, you cannot bake while driving. Be flexible and prioritize tasks based on order status and time efficiency. And also, you don't want your customer to wait for too long and you want the pizza arrive hot.

    Actions you can take: 
    - Confirm order
    - Start prepare order
    - Start deliver order
    - Finishing delivering order
    - Cancel order (only under special circumstances)
    - Do nothing (when you have nothing to do but wait for the pizza to be baked or you can't do anything because you're on the way to deliver)
    
    You can start prepare an order right after confirming it because confirming doesn't take long. 
    However, you can't start delivering right after preparing because baking the pizza take times.

    Conditions for Canceling an Order
    Cancel an order only if necessary, such as:  
    - Order overload: Too many orders at once, making it impossible to fulfill all.  
    - Unavoidable issues: Equipment failure making baking pizza impossible or extreme weather conditions making delivery impossible.  

    Below is the latest state of all orders:
    {latest_state}

    Based on your time and work management, decide the next action(s) in the next 10 minutes from {start_time} to {end_time} and output:  
    - **order_id**: The order being processed  
    - **action**: The action you are taking  
    - **timestamp**: The time at which you take the action (The timestamp should be within the given time frame)

    If handling multiple orders, output each action separately in the same format."""

    prompt = PromptTemplate.from_template(pizzeria_prompt)
    structured_llm = llm.with_structured_output(PizzeriaOutput)
    pizzeria_agent = prompt | structured_llm

    latest_state = get_latest_states(system_state)
    output = pizzeria_agent.invoke({'latest_state': latest_state, 'start_time': start_time, 'end_time': end_time})
    return output

### May be replace start_time with NOW to make more sense

### Running Simulation

In [ ]:
start_time = 